# Run 5 - Calcium network, synchronization & cell-assembly analysis

Post-Suite2p analysis of `F/Fneu/spks/iscell/ops`, structured around the recent
calcium-imaging methods literature (2021-2026).

**Event inference.** Events are detected on neuropil-corrected **dF/F0** (conservative,
validated), with each event's amplitude read in dF/F0 units; the deconvolved
**spks** (Suite2p OASIS) event rate is reported per cell as a cross-reference.
For cross-dataset-comparable spike inference, **CASCADE** (Rupprecht et al.,
*Nat. Neurosci.* 2021) is the recommended upgrade - a guarded hook is included
below; the notebook runs fully without it. Event-detection choice materially
affects synchrony conclusions (Wang et al., *Front. Neurosci.* 2021), so the
method here is explicit and all synchrony metrics ride on the same event trains.

**Synchronization (multi-metric, since the field is unsettled):**
1. **STTC** (Cutts & Eglen, 2014) - rate-independent pairwise synchrony (primary).
2. **Pearson** on dF/F0 (secondary, rate-dependent).
3. **Population coupling** (Okun et al., 2015) - each cell vs the population.
4. **Functional-connectivity graph** - edges = STTC above a circular-shift null.
5. **Cell assemblies** (PCA + Marchenko-Pastur + ICA; Lopes-dos-Santos 2013,
   as used for cortical-ensemble analysis, *eLife* 2021).
6. **Network bursts** with a shuffle-based significance threshold.

**Output:** one workbook `run5_event_metrics.xlsx`
(Summary / Per_Cell / Network_Bursts / Synchrony / Assemblies). No plots.

In [ ]:
# ============================================================
# IMPORTS + SETTINGS
# ============================================================
import numpy as np, pandas as pd
from pathlib import Path
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d
from IPython.display import display
import sys
sys.path.insert(0, "/Users/Samuel/Summer Research")   # for pipeline_fixes
import pipeline_fixes as pf

suite2p_folder = Path(".")
start_sec, end_sec = 0, None
fps_override = None
use_only_cells = True
neuropil_coeff = None   # set to 0.7 if ops['neucoeff'] is None

# --- dF/F0 baseline: rolling low-percentile (Fix 1 / item E) ---
baseline_percentile = 8       # unified F0 percentile (same as CASCADE input)
baseline_window_sec = None    # None = auto min(30, dur/4) + exp-detrend on short clips (fixed in pipeline_fixes)
baseline_step_sec = 1.0

smooth_window_sec = 0.5
threshold_std = 3.0
min_prominence_std = 1.0
min_distance_sec = 1.0

sttc_window_sec = 0.5          # +/- coincidence window for STTC
n_sttc_shuffles = 500          # circular-shift surrogates (mean-STTC null + per-pair FDR)
fdr_q = 0.05                   # BH-FDR level for per-pair functional connectivity (Fix 2)

assembly_bin_sec = 0.5         # bin width for cell-assembly detection
assembly_member_sd = 2.0       # member cell if ICA loading > mean + this*SD
assembly_min_bins = 50         # assembly guard (item D): need >= this many time-bins
assembly_max_NT = 0.2          #   AND N_active/T <= this, else skip as underpowered

coactivity_window_sec = 1.0
network_min_distance_sec = 2.0
network_fixed_threshold = 0.20
n_network_shuffles = 200
network_sig_percentile = 99

rng = np.random.default_rng(0)
n_top_cells_to_show = 20
output_prefix = "RUNID_metrics"   # <-- set per run; ONE combined workbook
output_xlsx = suite2p_folder / f"{output_prefix}.xlsx"
SAVE_OUTPUTS = True
SAVE_QC_PLOT = True            # rolling-baseline QC PNG (item G)

USE_CASCADE_IF_PRESENT = True
cascade_file = suite2p_folder / "cascade_spike_prob.npy"
# CASCADE event threshold. None = DATA-DRIVEN (median + k_mad*MAD of THIS recording's own
# spike-prob noise floor, via pipeline_fixes.cascade_events). Replaces the arbitrary 0.3 floor,
# which keeps almost nothing on sparse GCaMP8 spike-prob (on our 45 Hz jGCaMP8s test it maxed ~0.66). Set a float to
# force a fixed threshold (e.g. cascade_threshold = 0.3 to reproduce the old behaviour).
cascade_threshold = None
cascade_k_mad = 3.0            # data-driven: keep peaks >= median + k_mad*MAD above noise floor
cascade_min_prob = 0.05       # data-driven floor so a near-silent recording stays silent

# CASCADE spike extraction method for the event train that drives STTC/assemblies/bursts:
#   "discrete"  = CASCADE authors' calibrated, threshold-free spike inference
#                 (cascade2p.utils_discrete_spikes; Rupprecht et al. 2021). RECOMMENDED --
#                 rate/indicator-invariant, so recordings taken at different acquisition rates
#                 (whatever you set on the scope, per recording) are directly comparable.
#                 Requires the Cascade repo + model config.yaml.
#   "threshold" = peak-pick spike-prob at a data-driven threshold (pipeline_fixes.cascade_events);
#                 fast fallback, degenerates to cascade_min_prob on sparse recordings.
cascade_spike_method = "discrete"
cascade_random_seed = 0        # seed the discrete-spike Monte-Carlo sampler for reproducible trains


In [ ]:
# ============================================================
# LOAD
# ============================================================
F     = np.load(suite2p_folder/"F.npy",     allow_pickle=True)
Fneu  = np.load(suite2p_folder/"Fneu.npy",  allow_pickle=True)
spks  = np.load(suite2p_folder/"spks.npy",  allow_pickle=True).astype(np.float32)
iscell= np.load(suite2p_folder/"iscell.npy",allow_pickle=True)
ops   = np.load(suite2p_folder/"ops.npy",   allow_pickle=True).item()
n_rois_original, n_frames = F.shape
fps = float(fps_override) if fps_override else float(ops.get("fs", np.nan))
if not np.isfinite(fps) or fps<=0: raise ValueError("Set fps_override; ops['fs'] invalid.")
if neuropil_coeff is None: neuropil_coeff = float(ops.get("neucoeff", 0.7))
print(f"Loaded | ROIs {n_rois_original} | frames {n_frames} | fps {fps} | neuropil {neuropil_coeff}")

In [ ]:
# ============================================================
# SELECT CELLS + dF/F0  (rolling low-percentile baseline; Fix 1)
# ============================================================
cell_mask = iscell[:,0].astype(bool) if use_only_cells else np.ones(n_rois_original,bool)
roi_original_indices = np.where(cell_mask)[0]
spks_cells = spks[cell_mask]
n_cells = spks_cells.shape[0]
# Rolling baseline: F0(t) = sliding low percentile -> removes photobleaching drift.
dff_full, F0_full = pf.rolling_baseline_dff(F[cell_mask], Fneu[cell_mask], fs=fps,
    percentile=baseline_percentile, window_sec=baseline_window_sec,
    step_sec=baseline_step_sec, neuropil_coeff=neuropil_coeff)
F_corr = F[cell_mask] - neuropil_coeff*Fneu[cell_mask]   # kept for the QC plot
dff = np.nan_to_num(dff_full, nan=0.0)
_bw = "auto min(30,dur/4)" if baseline_window_sec is None else f"{baseline_window_sec:.0f}s"
print(f"Cells analyzed: {n_cells} | baseline: rolling {_bw} @ p{baseline_percentile}")


In [ ]:
# ============================================================
# ANALYSIS WINDOW
# ============================================================
start_frame = max(0, int(start_sec*fps))
end_frame = n_frames if end_sec is None else min(n_frames, int(end_sec*fps))
if end_frame<=start_frame: raise ValueError("end_frame must exceed start_frame.")
spks_win = spks_cells[:, start_frame:end_frame]
dff_win  = dff[:, start_frame:end_frame]
n_win = end_frame-start_frame
time_sec = np.arange(start_frame,end_frame)/fps
duration_sec = n_win/fps; duration_min = duration_sec/60
print(f"Window frames {start_frame}-{end_frame} ({duration_sec:.1f}s = {duration_min:.2f} min)")

In [ ]:
# ============================================================
# EVENT DETECTION  (CASCADE calibrated spikes if present, else dF/F0)  + spks cross-ref
# ============================================================
# dF/F0 is always computed: it supplies event AMPLITUDE (physiological units) and the
# secondary Pearson metric. The EVENT TRAINS that drive STTC / assemblies / network
# bursts come from CASCADE spike-probabilities when cascade_spike_prob.npy is present,
# otherwise from dF/F0 peak detection.
sw = int(round(smooth_window_sec*fps))
dff_smooth = uniform_filter1d(dff_win, sw, axis=1, mode="constant", cval=0.0) if sw>1 else dff_win.copy()
dff_baseline = np.nanmedian(dff_smooth, axis=1)

# --- choose the event signal ---
# CASCADE path uses pipeline_fixes.load_cascade_output (reads cascade_meta.json for the TRUE
# spike-prob rate -- may differ from ops['fs'] if run_cascade resampled to the model rate) and
# pipeline_fixes.cascade_events (data-driven threshold; replaces the arbitrary 0.3 floor).
use_cascade = USE_CASCADE_IF_PRESENT and cascade_file.exists()
cascade_fps = fps
if use_cascade:
    cprob_full, cascade_fps, cascade_meta = pf.load_cascade_output(str(suite2p_folder))
    if cprob_full.shape[0] != n_cells:
        print(f"CASCADE file has {cprob_full.shape[0]} rows but {n_cells} cells; using dF/F0 instead.")
        use_cascade = False
    elif abs(cascade_fps - fps) > 0.5:
        # spike_prob was emitted at the model rate; frame windows/rates must use cascade_fps.
        print(f"NOTE: CASCADE spike_prob is at {cascade_fps} Hz (model rate), not the folder's {fps} Hz. "
              f"Using {cascade_fps} Hz for CASCADE event timing.")
if use_cascade:
    # crop to the analysis window IN CASCADE FRAMES (cprob may be a different length than F).
    c_start = int(round(start_sec*cascade_fps))
    c_end   = cprob_full.shape[1] if end_sec is None else min(cprob_full.shape[1], int(round(end_sec*cascade_fps)))
    cprob_raw = cprob_full[:, c_start:c_end]          # keeps NaN edges (discrete-spike inference needs them)
    cprob = np.nan_to_num(cprob_raw, nan=0.0)
    detect = cprob; event_source = "CASCADE spike-prob (calibrated)"
    det_fps = cascade_fps
else:
    cprob = None; detect = dff_smooth; event_source = "dF/F0"; det_fps = fps
# number of frames and rate of the EVENT signal (CASCADE may be resampled to the model rate;
# dF/F0 path == F frames at fps). Downstream STTC/assembly/network timing uses these.
det_n_frames = detect.shape[1]

# per-cell baseline + robust noise on the detection signal (used for dF/F0 path + reporting)
det_base = np.nanmedian(detect, axis=1)
_dmad = np.nanmedian(np.abs(detect - det_base[:,None]), axis=1)
det_sigma = np.where(_dmad>0, 1.4826*_dmad, np.nanstd(detect, axis=1))
det_sigma = np.where(det_sigma>0, det_sigma, 1e-12)
min_dist = max(1, int(round(min_distance_sec*det_fps)))

# _want_discrete: try the calibrated discrete-spike method; if the cascade2p package or the
# model config.yaml is unavailable, fall back to the threshold method so the notebook still runs.
_want_discrete = use_cascade and cascade_spike_method == "discrete"
if _want_discrete:
    # PRIMARY: CASCADE authors' calibrated, threshold-free discrete spike inference
    # (cascade2p.utils_discrete_spikes; Rupprecht et al. 2021). Spike count per active region
    # = integrated probability mass; reads sampling_rate/smoothing from the model config, so
    # trains from different indicators/rates are directly comparable. Seeded for reproducibility.
    _model = cascade_meta.get("model")
    try:
        _dsp = pf.cascade_discrete_spikes(cprob_raw, _model, random_seed=cascade_random_seed)
        cascade_event_frames = _dsp["event_frames"]
        cascade_used_threshold = np.nan   # threshold-free
        event_source = "CASCADE discrete spikes (calibrated)"
        print(f"CASCADE discrete spikes (model {_model}): {int(_dsp['n_spikes'].sum())} spikes "
              f"across {_dsp['n_active']} cells")
        height_thr = np.zeros(n_cells); prom_thr = np.zeros(n_cells)
    except Exception as _e:
        # cascade2p not importable, or model config.yaml missing -> degrade to threshold method.
        print(f"WARNING: discrete-spike inference unavailable ({type(_e).__name__}: {_e}); "
              f"falling back to threshold method (cascade_events). "
              f"Install cascade2p + the model folder to enable the calibrated method.")
        _want_discrete = False
if use_cascade and not _want_discrete:
    # FALLBACK: threshold on spike-prob (pipeline_fixes.cascade_events). Reports when the
    # min_prob floor is what set the threshold (median+k_mad*MAD collapses on sparse data).
    _cev = pf.cascade_events(detect, det_fps, threshold=cascade_threshold,
                             k_mad=cascade_k_mad, min_prob=cascade_min_prob,
                             min_distance_sec=min_distance_sec)
    cascade_event_frames = _cev["event_frames"]
    cascade_used_threshold = _cev["threshold"]
    _fb = " (min_prob floor binding -- NOT data-adaptive here)" if _cev.get("floor_binding") else ""
    print(f"CASCADE threshold = {cascade_used_threshold:.4f} spike-prob{_fb}")
    height_thr = np.full(n_cells, cascade_used_threshold)   # for reporting only
    prom_thr = np.zeros(n_cells)
elif not use_cascade:
    cascade_event_frames = None
    height_thr = det_base + threshold_std*det_sigma
    prom_thr = min_prominence_std*det_sigma

# Calibrated per-cell spike RATE (spikes/min). Uses pipeline_fixes.cascade_spike_rate on the
# NaN-preserving spike-prob so the rate divides by VALID (inferred) seconds, not the full
# window -- CASCADE leaves ~receptive-field frames NaN at each edge (18% of a 10 s 45 Hz
# recording), and recordings differ in length, so dividing by full duration biases rate low
# by a length-dependent amount and would contaminate WT-vs-KCNT1 comparisons.
if use_cascade:
    _rate_ps, _tot_spikes = pf.cascade_spike_rate(cprob_raw, det_fps)   # per second, valid window
    cascade_rate_per_min = _rate_ps * 60.0

# spks cross-reference threshold (Run 4 style; complementary)
_sm = np.median(spks_win, axis=1, keepdims=True)
_sd = np.median(np.abs(spks_win-_sm), axis=1); _ss = np.std(spks_win, axis=1)
spks_noise = np.where(_sd>0, _sd/0.6745, _ss); spks_noise = np.where(spks_noise>0, spks_noise, 1e-12)
spks_height = _sm[:,0] + threshold_std*spks_noise

event_frames=[]; per_cell_rows=[]
for i in range(n_cells):
    if use_cascade:
        peaks = cascade_event_frames[i]            # data-driven CASCADE events (pipeline_fixes)
    else:
        peaks,_ = find_peaks(detect[i], height=height_thr[i], prominence=max(prom_thr[i],1e-9), distance=min_dist)
    event_frames.append(peaks); n_ev=len(peaks)
    # amplitude always read from dF/F0 (physiological units). Map CASCADE frames -> dF/F frames
    # when the rates differ, so amplitude is sampled at the right time.
    if n_ev:
        amp_idx = peaks if not use_cascade else np.clip(np.round(peaks*(fps/det_fps)).astype(int), 0, dff_smooth.shape[1]-1)
        amps = dff_smooth[i][amp_idx]-dff_baseline[i]
    else:
        amps = np.array([])
    ibi = np.diff(peaks)/det_fps if n_ev>=2 else np.array([])
    spk_pk,_ = find_peaks(spks_win[i], height=spks_height[i],
                          prominence=min_prominence_std*spks_noise[i], distance=min_dist)
    row = {
        "cell_number_in_analysis": i, "original_suite2p_roi_index": roi_original_indices[i],
        "active": n_ev>=1, "n_events": n_ev,
        "event_rate_per_min": n_ev/duration_min if duration_min else np.nan,
        "mean_event_amplitude_dff": np.nanmean(amps) if n_ev else np.nan,
        "median_event_amplitude_dff": np.nanmedian(amps) if n_ev else np.nan,
        "max_event_amplitude_dff": np.nanmax(amps) if n_ev else np.nan,
        "mean_inter_event_interval_sec": np.nanmean(ibi) if ibi.size else np.nan,
        "spks_event_rate_per_min": len(spk_pk)/duration_min if duration_min else np.nan,
        "trace_noise_robust_std": det_sigma[i],
    }
    if use_cascade:
        row["cascade_inferred_spikes_per_min"] = float(cascade_rate_per_min[i])   # valid-seconds normalized (NaN-edge corrected)
    per_cell_rows.append(row)
per_cell_metrics = pd.DataFrame(per_cell_rows)
active_mask = per_cell_metrics["active"].to_numpy()
active_idx = np.where(active_mask)[0]; n_active=int(active_mask.sum())
percent_active = 100*n_active/n_cells if n_cells else np.nan
total_events = int(per_cell_metrics["n_events"].sum())
active_event_frames = [event_frames[i] for i in active_idx]
print(f"Event signal: {event_source} | active {n_active}/{n_cells} ({percent_active:.1f}%) | total events {total_events}")

In [ ]:
# ============================================================
# SYNCHRONY 1: STTC (primary) + shuffle null -> z-score & empirical p
# ============================================================
dt = max(1, int(round(sttc_window_sec*det_fps)))
sttc = pf.sttc_matrix(active_event_frames, det_n_frames, dt)
iu = np.triu_indices(n_active, 1)
sttc_pairs = sttc[iu]; sttc_pairs = sttc_pairs[np.isfinite(sttc_pairs)]
mean_sttc = float(np.mean(sttc_pairs)) if sttc_pairs.size else np.nan
median_sttc = float(np.median(sttc_pairs)) if sttc_pairs.size else np.nan
sttc_per_cell = np.nanmean(sttc, axis=1) if n_active else np.array([])

chance_means=[]
for _ in range(n_sttc_shuffles):
    sh=[np.sort((fr+rng.integers(1,det_n_frames))%det_n_frames) for fr in active_event_frames]
    Ms=pf.sttc_matrix(sh, det_n_frames, dt); v=Ms[iu]; v=v[np.isfinite(v)]
    if v.size: chance_means.append(v.mean())
chance_mean_sttc = float(np.mean(chance_means)) if chance_means else np.nan
chance_sd_sttc = float(np.std(chance_means)) if len(chance_means)>1 else np.nan
chance_means_arr = np.asarray(chance_means, float)
sttc_p_emp = float((np.sum(chance_means_arr >= mean_sttc)+1)/(chance_means_arr.size+1)) if chance_means_arr.size else np.nan
sttc_z = (mean_sttc-chance_mean_sttc)/chance_sd_sttc if chance_sd_sttc and chance_sd_sttc>0 else np.nan
sttc_excess = mean_sttc - chance_mean_sttc   # effect size (item C): mean above chance
print(f"STTC mean {mean_sttc:.4f} (chance {chance_mean_sttc:.4f} +/- {chance_sd_sttc:.4f}, z {sttc_z:.1f}, p {sttc_p_emp:.3g})")

# --- STTC Δt-sensitivity profile (Cutts & Eglen 2014: STTC should be read AS A FUNCTION
# of the coincidence window, not at one Δt) ---------------------------------------------
# The primary metrics above use the single window sttc_window_sec. But tight synchrony
# (tens of ms) vs loose synchrony (hundreds of ms - seconds) is biologically distinct, and
# WT vs KCNT1 can differ in WHICH timescale they synchronize on. We profile mean pairwise
# STTC and its shuffle z across a range of windows so that comparison is visible. This is
# ADDITIVE -- it does not change mean_sttc / sttc_z / sttc_p_emp reported above.
sttc_dt_windows_sec = [0.02, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0]
_prof_rows=[]
for _w in sttc_dt_windows_sec:
    _dt = max(1, int(round(_w*det_fps)))
    _M = pf.sttc_matrix(active_event_frames, det_n_frames, _dt)
    _v = _M[iu]; _v = _v[np.isfinite(_v)]
    _obs = float(np.mean(_v)) if _v.size else np.nan
    _ch=[]
    for _ in range(min(n_sttc_shuffles, 200)):   # lighter null for the sweep
        _sh=[np.sort((fr+rng.integers(1,det_n_frames))%det_n_frames) for fr in active_event_frames]
        _Ms=pf.sttc_matrix(_sh, det_n_frames, _dt); _vv=_Ms[iu]; _vv=_vv[np.isfinite(_vv)]
        if _vv.size: _ch.append(_vv.mean())
    _cm=float(np.mean(_ch)) if _ch else np.nan
    _cs=float(np.std(_ch)) if len(_ch)>1 else np.nan
    _z=(_obs-_cm)/_cs if (_cs and _cs>0) else np.nan
    _prof_rows.append({"dt_window_sec":_w, "dt_frames":_dt, "mean_STTC":_obs,
                       "chance_mean_STTC":_cm, "STTC_excess_over_chance":(_obs-_cm),
                       "STTC_z_vs_chance":_z})
sttc_dt_profile = pd.DataFrame(_prof_rows)
_exc = sttc_dt_profile["STTC_excess_over_chance"] if len(sttc_dt_profile) else pd.Series(dtype=float)
_pk = sttc_dt_profile.loc[_exc.idxmax(), "dt_window_sec"] if _exc.notna().any() else np.nan
print(f"STTC Δt profile: peak synchrony excess at Δt={_pk}s "
      f"(profiled {list(sttc_dt_windows_sec)}s)")


In [ ]:
# ============================================================
# SYNCHRONY 2: PEARSON on smoothed dF/F0 (secondary)
# ============================================================
at = dff_smooth[active_mask]; good = np.nanstd(at,axis=1)>1e-12
if good.sum()>=2:
    cc = np.corrcoef(at[good]); pr = cc[np.triu_indices_from(cc,1)]; pr=pr[np.isfinite(pr)]
    mean_pearson=float(pr.mean()); median_pearson=float(np.median(pr))
else: mean_pearson=median_pearson=np.nan
print(f"Pearson mean {mean_pearson:.4f}")

In [ ]:
# ============================================================
# SYNCHRONY 3+5: POPULATION COUPLING (Okun) + CELL ASSEMBLIES (guarded; item D)
# ============================================================
bf = max(1, int(round(assembly_bin_sec*det_fps))); nbins = det_n_frames//bf
Z = np.zeros((n_active, nbins))
if use_cascade:
    for r,i in enumerate(active_idx):
        Z[r] = cprob[i, :nbins*bf].reshape(nbins, bf).sum(1)
else:
    for r,i in enumerate(active_idx):
        ev = event_frames[i]; ev = ev[ev < nbins*bf]
        np.add.at(Z[r], ev//bf, 1.0)
Zz = (Z - Z.mean(1,keepdims=True))/(Z.std(1,keepdims=True)+1e-12)

pop = Zz.sum(0)
population_coupling = np.array([np.corrcoef(Zz[r], pop-Zz[r])[0,1] for r in range(n_active)])
mean_pop_coupling = float(np.nanmean(population_coupling)) if n_active else np.nan

def fastica(Xw, k, iters=300, seed=0):
    r=np.random.default_rng(seed); W=np.zeros((k, Xw.shape[0]))
    for c in range(k):
        w=r.normal(size=Xw.shape[0]); w/=np.linalg.norm(w)
        for _ in range(iters):
            g=np.tanh(w@Xw); w1=(Xw*g).mean(1)-(1-g**2).mean()*w
            w1-=W[:c].T@(W[:c]@w1); w1/=np.linalg.norm(w1)
            if np.abs(np.abs(w1@w)-1)<1e-6: w=w1; break
            w=w1
        W[c]=w
    return W

# GUARD: Marchenko-Pastur needs many more time-bins than cells (T >> N).
N,T = Zz.shape
assembly_power_ok = pf.assembly_power_ok(n_active, nbins, min_bins=assembly_min_bins, max_ratio=assembly_max_NT)
assembly_rows=[]; assembly_member_of = -np.ones(n_active, int)
if not assembly_power_ok:
    n_assemblies = np.nan; mp_lambda = np.nan
    print(f"Population coupling mean {mean_pop_coupling:.3f} | assemblies: UNDERPOWERED - skipped "
          f"(N={n_active}, T={nbins} bins; need T>={assembly_min_bins} and N/T<={assembly_max_NT})")
else:
    Ccorr = Zz@Zz.T/T
    eigval, eigvec = np.linalg.eigh(Ccorr); eigval=eigval[::-1]; eigvec=eigvec[:,::-1]
    mp_lambda = (1+np.sqrt(N/T))**2
    n_assemblies = int((eigval>mp_lambda).sum())
    if n_assemblies>=1:
        d=eigval[:n_assemblies]; E=eigvec[:,:n_assemblies]
        Wh=(E/np.sqrt(d+1e-12)).T
        patterns = fastica(Wh@Zz, n_assemblies) @ Wh
        patterns /= np.linalg.norm(patterns, axis=1, keepdims=True)
        for a in range(n_assemblies):
            p = patterns[a]
            if abs(p.min())>abs(p.max()): p=-p; patterns[a]=p
            members = np.where(p > p.mean()+assembly_member_sd*p.std())[0]
            assembly_member_of[members] = a
            Pm = np.outer(p,p); np.fill_diagonal(Pm,0.0)
            R = np.einsum('it,ij,jt->t', Zz, Pm, Zz)
            assembly_rows.append({
                "assembly": a+1, "n_member_cells": int(len(members)),
                "peak_activation_strength": float(R.max()) if R.size else np.nan,
                "n_strong_activations": int((R > R.mean()+3*R.std()).sum()),
                "top_member_roi_indices": ",".join(map(str,
                    roi_original_indices[active_idx[members[np.argsort(p[members])[::-1][:8]]]])) if len(members) else "",
            })
    print(f"Population coupling mean {mean_pop_coupling:.3f} | assemblies (MP) {n_assemblies}")
assemblies = pd.DataFrame(assembly_rows)


In [ ]:
# ============================================================
# SYNCHRONY 4: FUNCTIONAL-CONNECTIVITY GRAPH  (per-pair STTC + BH-FDR; Fix 2)
# ============================================================
# Old rule (edge = STTC > 99th pct of null) keeps ~1% of pairs BY CONSTRUCTION.
# Replaced with per-pair empirical p + Benjamini-Hochberg FDR across all pairs.
net = pf.fdr_connectivity(active_event_frames, det_n_frames, dt, n_shuffles=n_sttc_shuffles, q=fdr_q, rng=rng)
functional_degree = net["degree"]
n_edges = net["n_edges"]; edge_density = net["density"]; mean_degree = net["mean_degree"]
n_edges_null_expected = net["null_edges_expected"]; fdr_min_p = net["min_p"]
frac_pairs_sig = (n_edges/net["n_pairs"]) if net["n_pairs"] else np.nan
print(f"Graph (FDR q={fdr_q}) | edges {n_edges} (old 99pct-rule would falsely give ~{n_edges_null_expected:.0f}) "
      f"| density {edge_density:.4f} | mean degree {mean_degree:.2f} | p-floor {fdr_min_p:.4f}")


In [ ]:
# ============================================================
# NETWORK BURSTS + shuffle-based significance
# ============================================================
half = max(1, int(round((coactivity_window_sec*det_fps)/2)))
covered = np.zeros((n_active, det_n_frames), bool)
for r,i in enumerate(active_idx):
    for p in event_frames[i]: covered[r, max(0,p-half):min(det_n_frames,p+half+1)] = True
coactive_count = covered.sum(0)
coactive_fraction = coactive_count/n_active if n_active else np.zeros(det_n_frames)
# Detection-timebase seconds: network-burst frames index the EVENT signal (det_fps,
# det_n_frames), which differs from the fluorescence timebase (fps) when the spike-prob
# was resampled to the model rate (e.g. SS10 jGCaMP8f resampled to 100 Hz). Indexing the
# F-based time_sec here would be out of range / off by the resample ratio. For SS9 the two
# timebases are identical, so this is a no-op there.
det_time_sec = start_frame/fps + np.arange(det_n_frames)/det_fps

shuffle_peaks=np.empty(n_network_shuffles); tvec=np.arange(det_n_frames)
for s in range(n_network_shuffles):
    shifts=rng.integers(0,det_n_frames,n_active)
    frac=covered[np.arange(n_active)[:,None], (tvec[None,:]-shifts[:,None])%det_n_frames].sum(0)/n_active
    shuffle_peaks[s]=frac.max()
sig_threshold=float(np.percentile(shuffle_peaks, network_sig_percentile))
net_min_dist=max(1,int(round(network_min_distance_sec*det_fps)))
fixed_burst_frames,_=find_peaks(coactive_fraction, height=network_fixed_threshold, distance=net_min_dist)
sig_burst_frames,_=find_peaks(coactive_fraction, height=max(sig_threshold,1e-9), distance=net_min_dist)
network_bursts=pd.DataFrame({
    "network_burst_number": np.arange(1,len(sig_burst_frames)+1),
    "time_sec": det_time_sec[sig_burst_frames],
    "coactive_count": coactive_count[sig_burst_frames],
    "coactive_fraction": coactive_fraction[sig_burst_frames]})
frac_cells_in_bursts=float(covered[:,sig_burst_frames].any(1).mean()) if len(sig_burst_frames) else np.nan
print(f"Network bursts | fixed {len(fixed_burst_frames)} | shuffle-sig {len(sig_burst_frames)} (thr {sig_threshold:.3f})")

In [ ]:
# ============================================================
# BASELINE QC PLOT (item G) - raw F with rolling F0 under it
# ============================================================
if SAVE_QC_PLOT:
    qc_png = suite2p_folder / f"{output_prefix}_baseline_qc.png"
    try:
        pf.baseline_qc_plot(F_corr, F0_full, fps, str(qc_png), n_cells=6)
        print("Saved baseline QC:", qc_png)
    except Exception as e:
        print("QC plot skipped:", e)


In [ ]:
# ============================================================
# SUMMARY (sectioned)
# ============================================================
act=per_cell_metrics[per_cell_metrics["active"]]; rate=act["event_rate_per_min"]; amp=act["mean_event_amplitude_dff"]
def P(s,q): return float(np.nanpercentile(s,q)) if len(s) else np.nan
rows=[
 ("Recording","n_original_suite2p_rois",n_rois_original),
 ("Recording","n_cells_analyzed",n_cells),
 ("Recording","n_active_cells",n_active),
 ("Recording","percent_active_cells",percent_active),
 ("Recording","fps",fps),("Recording","duration_min",duration_min),
 ("Events","event_signal_source",event_source),
 ("Events","total_events_detected",total_events),
 ("Events","mean_events_per_active_cell",float(act["n_events"].mean()) if n_active else np.nan),
 ("Events (deconvolved cross-ref)","mean_spks_event_rate_per_min",float(act["spks_event_rate_per_min"].mean()) if n_active else np.nan),
 ("Events (CASCADE)","mean_inferred_spikes_per_min",float(act["cascade_inferred_spikes_per_min"].mean()) if ("cascade_inferred_spikes_per_min" in act.columns and n_active) else np.nan),
 ("Event rate (active,/min)","mean",float(rate.mean()) if n_active else np.nan),
 ("Event rate (active,/min)","median",float(rate.median()) if n_active else np.nan),
 ("Event rate (active,/min)","p25",P(rate,25)),("Event rate (active,/min)","p75",P(rate,75)),
 ("Event amplitude (active,dF/F0)","mean",float(amp.mean()) if n_active else np.nan),
 ("Event amplitude (active,dF/F0)","median",float(amp.median()) if n_active else np.nan),
 ("Inter-event interval (active,sec)","mean",float(act["mean_inter_event_interval_sec"].mean()) if n_active else np.nan),
 ("Synchrony - STTC (primary)","mean_STTC",mean_sttc),
 ("Synchrony - STTC (primary)","median_STTC",median_sttc),
 ("Synchrony - STTC (primary)","chance_mean_STTC",chance_mean_sttc),
 ("Synchrony - STTC (primary)","STTC_z_vs_chance",sttc_z),
 ("Synchrony - STTC (primary)","STTC_p_empirical_vs_chance",sttc_p_emp),
 ("Synchrony - STTC (primary)","chance_sd_STTC",chance_sd_sttc),
 ("Synchrony - STTC (primary)","STTC_excess_over_chance",sttc_excess),
 ("Synchrony - STTC (primary)","n_cell_pairs",int(sttc_pairs.size)),
 ("Synchrony - Pearson (secondary)","mean_pearson_dff",mean_pearson),
 ("Population coupling (Okun)","mean_population_coupling",mean_pop_coupling),
 ("Functional connectivity graph (FDR)","n_edges",n_edges),
 ("Functional connectivity graph (FDR)","edge_density",edge_density),
 ("Functional connectivity graph (FDR)","mean_degree",mean_degree),
 ("Functional connectivity graph (FDR)","null_edges_expected_old_rule",n_edges_null_expected),
 ("Functional connectivity graph (FDR)","frac_pairs_FDR_significant",frac_pairs_sig),
 ("Functional connectivity graph (FDR)","fdr_q",fdr_q),
 ("Cell assemblies (PCA+MP+ICA)","n_assemblies",n_assemblies),
 ("Cell assemblies (PCA+MP+ICA)","power_ok_T_gg_N",assembly_power_ok),
 ("Cell assemblies (PCA+MP+ICA)","marchenko_pastur_lambda",float(mp_lambda)),
 ("Cell assemblies (PCA+MP+ICA)","assembly_bin_sec",assembly_bin_sec),
 ("Cell assemblies (PCA+MP+ICA)","n_cells_in_an_assembly",int((assembly_member_of>=0).sum())),
 ("Network bursts","fixed_threshold_fraction",network_fixed_threshold),
 ("Network bursts","n_bursts_fixed_threshold",len(fixed_burst_frames)),
 ("Network bursts","shuffle_significance_threshold",sig_threshold),
 ("Network bursts",f"n_bursts_shuffle_p{network_sig_percentile}",len(sig_burst_frames)),
 ("Network bursts","max_coactive_fraction",float(coactive_fraction.max()) if n_active else np.nan),
 ("Network bursts","fraction_cells_in_significant_bursts",frac_cells_in_bursts),
 ("Parameters","threshold_std",threshold_std),("Parameters","sttc_window_sec",sttc_window_sec),
 ("Parameters","coactivity_window_sec",coactivity_window_sec),("Parameters","neuropil_coeff",neuropil_coeff),
]
summary_table=pd.DataFrame(rows, columns=["Section","Metric","Value"])
print("Summary computed.")

In [ ]:
# ============================================================
# DISPLAY (organized tables, no plots)
# ============================================================
# Presentation only -- computes nothing and writes nothing. Styled tables need jinja2;
# when it (or an interactive frontend) is absent, degrade to a plain display/print so a
# "Run All" in a bare environment can NEVER halt here before the export cell runs.
def _f(v): return f"{v:.4f}" if isinstance(v,float) else str(v)
def _show(df, caption, fmt=None):
    print(f"\n=== {caption} ===")
    try:
        sty = df.style.hide(axis="index")
        if fmt: sty = sty.format(fmt)
        display(sty.set_caption(caption))
    except Exception:
        # jinja2 missing / no notebook frontend -> plain text, same numbers
        d = df.copy()
        if fmt:
            for col, spec in fmt.items():
                if col in d.columns:
                    d[col] = d[col].apply(lambda v: spec.format(v) if isinstance(v,(int,float)) else v)
        try: display(d)
        except Exception: print(d.to_string(index=False))

_s=summary_table.copy(); _s["Value"]=_s["Value"].apply(_f)
_show(_s, "Run 5 - Summary Metrics")

_top=(per_cell_metrics[["cell_number_in_analysis","original_suite2p_roi_index","active","n_events",
      "event_rate_per_min","mean_event_amplitude_dff"]]
      .sort_values("event_rate_per_min",ascending=False).head(n_top_cells_to_show))
_show(_top, f"Run 5 - Top {n_top_cells_to_show} Cells by Event Rate",
      {"event_rate_per_min":"{:.2f}","mean_event_amplitude_dff":"{:.4f}"})

if len(assemblies):
    _show(assemblies, f"Run 5 - Cell Assemblies ({n_assemblies} detected)",
          {"peak_activation_strength":"{:.1f}"})
if len(network_bursts):
    _show(network_bursts, "Run 5 - Significant Network Bursts",
          {"time_sec":"{:.2f}","coactive_fraction":"{:.3f}"})


In [ ]:
# ============================================================
# EXPORT: ONE COMBINED WORKBOOK  (friendly sheets + technical sheets)
# ============================================================
# Writes a single <output_prefix>.xlsx containing BOTH:
#   * beginner-friendly sheets: How to Read This / Key Numbers / Firing Rates /
#     Firing Together / Group Events / Teams  (renamed + interpreted, NEVER recomputed)
#   * technical sheets: Summary / Per_Cell / Synchrony / Network_Bursts / Assemblies
# The friendly sheets are built from the same in-memory results, so they always match.
try:
    from openpyxl.comments import Comment
    from openpyxl.styles import Font, Alignment, PatternFill
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False
    print("WARNING: openpyxl not installed in this environment -- the styled combined workbook\n"
          "         cannot be written. Install it with:  pip install openpyxl\n"
          "         Falling back to CSV sheets (same numbers, no styling) so results ARE written.")

def _num(v):
    try: return float(v)
    except (ValueError, TypeError): return v

# ---- technical dataframes ----
per_cell_out = per_cell_metrics.copy()
pc = np.full(n_cells, np.nan); pc[active_idx]=population_coupling
fd = np.full(n_cells, np.nan); fd[active_idx]=functional_degree
am = np.full(n_cells, -1, int); am[active_idx]=assembly_member_of
per_cell_out["population_coupling"]=pc
per_cell_out["functional_degree"]=fd
per_cell_out["assembly_id"]=np.where(am>=0, am+1, np.nan)
per_cell_out=per_cell_out.sort_values("event_rate_per_min",ascending=False)
synchrony=pd.DataFrame({
    "original_suite2p_roi_index":roi_original_indices[active_idx],
    "mean_STTC_to_population":sttc_per_cell,
    "population_coupling":population_coupling,
    "functional_degree":functional_degree,
    "assembly_id":np.where(assembly_member_of>=0, assembly_member_of+1, np.nan),
}).sort_values("mean_STTC_to_population",ascending=False)

# ---- headline numbers pulled from the in-memory Summary (never recompute) ----
pair = {(str(s),str(mt)):_num(v) for s,mt,v in summary_table.itertuples(index=False)}
m = {str(mt):_num(v) for s,mt,v in summary_table.itertuples(index=False)}
RATE="Event rate (active,/min)"
m["rate_mean"]=pair.get((RATE,"mean")); m["rate_median"]=pair.get((RATE,"median"))
m["rate_p25"]=pair.get((RATE,"p25")); m["rate_p75"]=pair.get((RATE,"p75"))

COLUMN_HELP = {
    "Firing Rates (per cell)": {
        "cell_id": "The cell's ID number in the raw Suite2p data (its ROI index). Use it to trace a cell back to the movie.",
        "firings_per_minute": "How many calcium events this cell had, per minute. This is its firing rate. Higher = more active.",
        "total_firings": "Total calcium events for this cell across the whole recording.",
        "one_firing_every_sec": "Average gap between this cell's events, in seconds. Lower = it fires more often. (Blank if it fired only once.)",
        "activity_level": "Plain label: 'very active' = top 10% by firing rate, 'above average' = faster than the middle cell, 'quiet' = slower.",
    },
    "Firing Together (per cell)": {
        "cell_id": "The cell's ID number in the raw Suite2p data (its ROI index).",
        "togetherness_score_0to1": "How reliably this cell fires at the SAME times as the rest of the network (STTC). 0 = random timing vs others, 1 = perfectly in step. Higher = more synchronized.",
        "follows_the_crowd_0to1": "How closely this cell tracks the overall population's activity (population coupling). Near 1 = mirrors the crowd, near 0 = does its own thing, negative = tends to go opposite.",
        "num_partner_cells": "How many other cells this one is significantly co-active with (its number of connections). Higher = more of a network hub.",
        "team_number": "Which co-firing team (cell assembly) this cell belongs to, if any. Blank = not a strong member of any team.",
    },
    "Group Events (bursts)": {
        "burst_number": "Index of the network burst - a moment when unusually many cells fired almost simultaneously.",
        "time_seconds": "When in the recording this burst happened, in seconds from the start.",
        "num_cells_firing_together": "How many cells were active during this burst.",
        "percent_of_network": "That count as a percentage of all active cells. Higher = a bigger, more network-wide burst.",
    },
    "Teams (assemblies)": {
        "team_number": "The team's ID (cell-assembly number).",
        "num_cells_on_team": "How many cells belong to this co-firing team.",
        "peak_team_strength": "The team's strongest single co-activation (unitless). Bigger = the members fired together more intensely at their peak moment.",
        "times_team_fired_strongly": "How many times during the recording this team activated strongly together.",
        "example_cell_ids": "A few of the team's member cell IDs (Suite2p ROI numbers), as examples.",
    },
}

def _style_header(ws, ncols, help_map=None):
    fill = PatternFill("solid", fgColor="DDE7F0")
    for j in range(1, ncols + 1):
        c = ws.cell(row=1, column=j)
        c.font = Font(bold=True); c.fill = fill
        c.alignment = Alignment(vertical="center", wrap_text=True)
        if help_map:
            note = help_map.get(str(c.value))
            if note:
                cm = Comment(note, "analysis"); cm.width = 320; cm.height = 160
                c.comment = cm

def _autowidth(ws, df):
    for j, col in enumerate(df.columns, 1):
        w = max([len(str(col))] + [len(str(x)) for x in df[col].head(200)]) if len(df) else len(str(col))
        ws.column_dimensions[ws.cell(row=1, column=j).column_letter].width = min(w + 2, 52)

# ---- friendly sheet dataframes ----
med = m.get("rate_median", np.nan)
def _row(measure, value, meaning): return {"Measure": measure, "Value": value, "What it means": meaning}
key = pd.DataFrame([
    _row("- RECORDING -","",""),
    _row("Cells tracked", int(m["n_cells_analyzed"]), "Every ROI Suite2p labeled as a cell in this recording."),
    _row("Cells that fired (active)", int(m["n_active_cells"]), "Cells with at least one calcium event. The rest stayed silent."),
    _row("Percent active", f'{m["percent_active_cells"]:.0f}%', "Share of tracked cells that fired at all."),
    _row("Recording length (min)", f'{m["duration_min"]:.1f}', "How long the analyzed movie is."),
    _row("- HOW FAST CELLS FIRE (on their own) -","",""),
    _row("Typical firing rate (median /min)", f'{med:.2f}' if med==med else "n/a", "The middle cell's events per minute. Half the cells fire faster, half slower."),
    _row("Typical cell fires about every (sec)", f'{(60/med):.1f}' if med else "n/a", "The same thing as a gap between events: the typical cell has an event about this often, in seconds."),
    _row("Average firing rate (mean /min)", f'{m["rate_mean"]:.2f}' if m["rate_mean"]==m["rate_mean"] else "n/a", "Mean events per minute. Higher than the median because a few busy cells pull it up."),
    _row("- HOW MUCH CELLS FIRE TOGETHER (synchronization) -","",""),
    _row("Togetherness (STTC, -1 to +1)", f'{m["mean_STTC"]:.3g}', "Average pairwise synchrony (STTC). +1 = cells fire together, 0 = independent (random timing), negative = they tend to avoid each other."),
    _row("Is that togetherness real? (z-score)", f'{m["STTC_z_vs_chance"]:.3g}', "How far the togetherness stands above pure chance, in standard deviations vs shuffled data. Below ~2 = could just be random (treat as not synchronized); above ~2-3 = unlikely to be chance; very large = unmistakably real."),
    _row("...as a p-value", f'{m.get("STTC_p_empirical_vs_chance", float("nan")):.3g}', "The same question as a probability: chance of seeing this much togetherness if the timing were random (from the shuffle test). Below 0.05 = statistically significant / real; above = could be chance."),
    _row("...and how big vs chance (excess)", f'{m.get("STTC_excess_over_chance", float("nan")):.3g}', "Effect SIZE: the togetherness minus the chance level. A result can be statistically real (low p) yet tiny (small excess). Compare recordings on this, not just the p-value."),
    _row("Follows-the-crowd (pop. coupling, 0-1)", f'{m["mean_population_coupling"]:.2f}', "How strongly the average cell tracks the whole population. Higher = more herd-like."),
    _row("Average partners per cell", f'{m["mean_degree"]:.0f}', "On average, how many other cells each one is significantly co-active with."),
    _row("Cell teams (assemblies)", (int(m["n_assemblies"]) if np.isfinite(m["n_assemblies"]) else "underpowered"), "Number of groups of cells that repeatedly fire together as a team."),
    _row("Cells that belong to a team", int(m["n_cells_in_an_assembly"]), "How many cells are strong members of at least one team."),
    _row("Network bursts", int(m.get("n_bursts_shuffle_p99", np.nan)) if m.get("n_bursts_shuffle_p99")==m.get("n_bursts_shuffle_p99") else 0, "Moments when unusually many cells fired almost simultaneously."),
    _row("Biggest burst (% of cells)", f'{m["max_coactive_fraction"]*100:.0f}%', "The largest fraction of the network that fired all at once."),
], columns=["Measure","Value","What it means"])

fr = per_cell_metrics[per_cell_metrics["active"]==True].copy()
fr["firings_per_minute"]=fr["event_rate_per_min"].round(2)
fr["total_firings"]=fr["n_events"].astype(int)
fr["one_firing_every_sec"]=fr["mean_inter_event_interval_sec"].round(1)
q=fr["event_rate_per_min"]; hi,mid=q.quantile(0.90),q.quantile(0.50)
fr["activity_level"]=q.apply(lambda v: "very active (top 10%)" if v>=hi else ("above average" if v>=mid else "quiet"))
firing_rates=(fr[["original_suite2p_roi_index","firings_per_minute","total_firings","one_firing_every_sec","activity_level"]]
    .rename(columns={"original_suite2p_roi_index":"cell_id"})
    .sort_values("firings_per_minute",ascending=False).reset_index(drop=True))

sy=synchrony.copy().rename(columns={"original_suite2p_roi_index":"cell_id",
    "mean_STTC_to_population":"togetherness_score_0to1","population_coupling":"follows_the_crowd_0to1",
    "functional_degree":"num_partner_cells","assembly_id":"team_number"})
sy["togetherness_score_0to1"]=sy["togetherness_score_0to1"].round(3)
sy["follows_the_crowd_0to1"]=sy["follows_the_crowd_0to1"].round(3)
sy=sy.sort_values("togetherness_score_0to1",ascending=False).reset_index(drop=True)

gb=network_bursts.copy().rename(columns={"network_burst_number":"burst_number","time_sec":"time_seconds",
    "coactive_count":"num_cells_firing_together","coactive_fraction":"fraction_of_network"})
if "fraction_of_network" in gb.columns:
    gb["percent_of_network"]=(gb["fraction_of_network"]*100).round(1); gb=gb.drop(columns=["fraction_of_network"])
if "time_seconds" in gb.columns: gb["time_seconds"]=gb["time_seconds"].round(1)

asm=assemblies.copy().rename(columns={"assembly":"team_number","n_member_cells":"num_cells_on_team",
    "peak_activation_strength":"peak_team_strength","n_strong_activations":"times_team_fired_strongly",
    "top_member_roi_indices":"example_cell_ids"})

data_sheets=[("Firing Rates (per cell)",firing_rates),("Firing Together (per cell)",sy),
             ("Group Events (bursts)",gb),("Teams (assemblies)",asm)]

guide_rows=[{"Sheet":"Key Numbers","Column / Item":"(whole sheet)",
             "What this number means":"The headline results. Read the 'What it means' column next to each number."}]
_sheet_intro={"Firing Rates (per cell)":"One row per active cell: how fast it fires on its own. Sorted busiest first.",
    "Firing Together (per cell)":"One row per active cell: how synchronized it is with the network. Sorted most-in-sync first.",
    "Group Events (bursts)":"One row per network burst: a moment many cells fired at once.",
    "Teams (assemblies)":"One row per cell team: groups of cells that repeatedly fire together."}
for sheet_name,df in data_sheets:
    guide_rows.append({"Sheet":sheet_name,"Column / Item":"(whole sheet)","What this number means":_sheet_intro[sheet_name]})
    for col in df.columns:
        guide_rows.append({"Sheet":sheet_name,"Column / Item":col,"What this number means":COLUMN_HELP[sheet_name].get(col,"")})
guide=pd.DataFrame(guide_rows,columns=["Sheet","Column / Item","What this number means"])

# ---- PROVENANCE stamp: authoritative record of what produced this workbook ----
# Concern-5 fix: no more guessing which of several sibling workbooks is current.
import time as _time
_cm = cascade_meta if ("cascade_meta" in dir() and isinstance(cascade_meta, dict)) else {}
provenance = pd.DataFrame([
    ("input_folder", str(Path(suite2p_folder).resolve())),
    ("output_prefix", str(output_prefix)),
    ("generated", _time.strftime("%Y-%m-%dT%H:%M:%S")),
    ("event_source", str(event_source)),
    ("acquisition_fps", f"{fps:g}"),
    ("detection_fps (det_fps)", f"{det_fps:g}"),
    ("cascade_model", _cm.get("model", "n/a (dF/F path)")),
    ("cascade_family", _cm.get("family", "n/a")),
    ("cascade_model_rate_hz", _cm.get("model_rate_hz", "n/a")),
    ("cascade_resampled", _cm.get("resampled", "n/a")),
    ("cascade_input_fps", _cm.get("input_fps", "n/a")),
    ("cascade_output_fps", _cm.get("output_fps", "n/a")),
    ("n_cells_analyzed", int(n_cells)),
    ("n_active_cells", int(n_active)),
    ("total_events", int(total_events)),
    ("cascade_ref", _cm.get("cascade_ref", "Rupprecht et al. 2021 Nat Neurosci")),
], columns=["field", "value"])

# ---- write ONE combined workbook: friendly sheets first, technical after ----
technical_sheets=[("Provenance",provenance),("Summary",summary_table),("Per_Cell",per_cell_out),("Synchrony",synchrony),
                  ("STTC_vs_dt",sttc_dt_profile if "sttc_dt_profile" in dir() else pd.DataFrame()),
                  ("Network_Bursts",network_bursts),("Assemblies",assemblies)]
_ALL_SHEETS = [("How to Read This",guide),("Key Numbers",key)] + data_sheets + technical_sheets

def _save_csv_fallback(prefix_path):
    # Write every sheet as an individual CSV into <prefix>_sheets/. Guarantees current
    # numbers land on disk even without openpyxl, so a re-run is never a silent no-op.
    outdir = Path(str(prefix_path).replace(".xlsx","") + "_sheets")
    outdir.mkdir(exist_ok=True)
    for name, df in _ALL_SHEETS:
        (df if len(df) else pd.DataFrame(columns=df.columns)).to_csv(
            outdir / (name.replace(" ","_").replace("/","-") + ".csv"), index=False)
    return outdir

def _write_provenance_txt(path):
    stamp = Path(str(path).replace(".xlsx","") + ".PROVENANCE.txt")
    with open(stamp, "w") as fh:
        fh.write("# Authoritative provenance for this metrics run.\n")
        fh.write("# If two workbooks disagree, the one whose .PROVENANCE.txt is newest and\n")
        fh.write("# whose input_folder is the native suite2p/ folder is current.\n\n")
        for f, v in provenance.itertuples(index=False):
            fh.write(f"{f}: {v}\n")
    return stamp

def save_all(path=output_xlsx):
    path=Path(path)
    _write_provenance_txt(path)
    if not HAS_OPENPYXL:
        return _save_csv_fallback(path)
    with pd.ExcelWriter(path, engine="openpyxl") as xl:
        guide.to_excel(xl,"How to Read This",index=False)
        _style_header(xl.sheets["How to Read This"],len(guide.columns)); _autowidth(xl.sheets["How to Read This"],guide)
        key.to_excel(xl,"Key Numbers",index=False)
        _style_header(xl.sheets["Key Numbers"],len(key.columns)); _autowidth(xl.sheets["Key Numbers"],key)
        for name,df in data_sheets:
            df.to_excel(xl,name,index=False)
            _style_header(xl.sheets[name],len(df.columns),COLUMN_HELP[name]); _autowidth(xl.sheets[name],df)
        for name,df in technical_sheets:
            (df if len(df) else pd.DataFrame(columns=df.columns)).to_excel(xl,name,index=False)
            ws=xl.sheets[name]
            for j,col in enumerate(df.columns,1):
                w=max([len(str(col))]+[len(str(x)) for x in df[col].head(200)]) if len(df) else len(str(col))
                ws.column_dimensions[ws.cell(row=1,column=j).column_letter].width=min(w+2,44)
        for _ws in xl.book.worksheets:
            for _row in _ws.iter_rows():
                for _cell in _row:
                    if isinstance(_cell.value, float):
                        _cell.number_format = "0.####"   # show decimals in all viewers (not 'General')
    return path
if SAVE_OUTPUTS:
    _saved = save_all()
    print(("Saved combined workbook: " if HAS_OPENPYXL else "openpyxl missing -- saved CSV sheets to: ") + str(_saved))
else:
    print("SAVE_OUTPUTS=False; call save_all().")


## Friendly 'Key Numbers' preview

In [ ]:
# ============================================================
# PREVIEW: friendly "Key Numbers" sheet
# ============================================================
# Display-only. Uses _show() (defined in the DISPLAY cell) so it degrades to plain text
# when jinja2 / a notebook frontend is absent, instead of crashing at the end of a run.
try:
    _show(key, "Key Numbers (beginner-friendly)")
except NameError:
    # _show not defined (cell run out of order) -> minimal inline fallback
    try: display(key)
    except Exception: print(key.to_string(index=False))
